In [2]:
!uv add -r rag-requirements.txt

Resolved 237 packages in 1ms
Checked 211 packages in 33ms


In [4]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv
logging.langsmith("SQL")

db = SQLDatabase.from_uri("sqlite:///data/manufacturing.db")
print(db.dialect)
print(db.get_usable_table_names())

LangSmith 추적을 시작합니다.
[프로젝트명]
SQL
sqlite
['alarms', 'employees', 'energy_usage', 'equipment', 'factories', 'maintenance_records', 'production_lines', 'production_orders', 'production_results', 'products', 'quality_inspections']


In [12]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
chain = create_sql_query_chain(llm, db)

generated_sql_query = chain.invoke({"question": "장비의 상태을 알려주세요"})
print(generated_sql_query.__repr__())

'Question: 장비의 상태을 알려주세요  \nSQLQuery: SELECT "equipment_name", "status" FROM "equipment" LIMIT 5;'


In [13]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
execute_query = QuerySQLDataBaseTool(db=db)
execute_query.invoke({"query": generated_sql_query})

'Error: (sqlite3.OperationalError) near "Question": syntax error\n[SQL: Question: 장비의 상태을 알려주세요  \nSQLQuery: SELECT "equipment_name", "status" FROM "equipment" LIMIT 5;]\n(Background on this error at: https://sqlalche.me/e/21/e3q8)'

In [16]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
execute_query = QuerySQLDataBaseTool(db=db)
write_query = create_sql_query_chain(llm, db)
chain = write_query | execute_query

chain.invoke({"question": "모터의 상태를 조회하세요"})

'Error: (sqlite3.OperationalError) near "Question": syntax error\n[SQL: Question: 모터의 상태를 조회하세요\nSQLQuery: \nSELECT "equipment_name", "status" \nFROM "equipment" \nWHERE "equipment_type" = \'Motor\' \nLIMIT 5;]\n(Background on this error at: https://sqlalche.me/e/21/e3q8)'

In [18]:
from operator import itemgetter

from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.1
)

sql_prompt = PromptTemplate.from_template("""
You are an expert SQLite query generator.

Generate a correct SQLite query using ONLY the provided database schema.

IMPORTANT RULES:

- Never invent table names or column names.

- Manufacturing defect rate is:

  defect_qty / (good_qty + defect_qty)

- For grouped defect rates such as shift, line, product, or date:

  SUM(defect_qty) * 100.0
  / NULLIF(SUM(good_qty + defect_qty), 0)

- NEVER use defect_qty / good_qty for defect rate.

- Do NOT use AVG(row_defect_rate) unless the user explicitly asks
  for the arithmetic average of individual row rates.

- production_results does NOT contain line_id.

  line_id must be obtained through:

  production_results.order_id
  -> production_orders.order_id
  -> production_orders.line_id

- Qualify ambiguous columns using table aliases.

- Avoid unnecessary JOINs.

- Use NULLIF for division.

- Return ONLY SQL.
- Do not use Markdown.

Example question:
주간조와 야간조의 평균 불량률을 비교해줘.

Example correct SQL:

SELECT
    pr.shift,
    ROUND(
        100.0 * SUM(pr.defect_qty)
        / NULLIF(SUM(pr.good_qty + pr.defect_qty), 0),
        2
    ) AS defect_rate_pct
FROM production_results AS pr
GROUP BY pr.shift;

Database schema:
{table_info}

Question:
{input}

Maximum rows if LIMIT is appropriate: {top_k}

SQLQuery:
"""
)

write_query = create_sql_query_chain(
    llm,
    db,
    prompt=sql_prompt
)

execute_query = QuerySQLDataBaseTool(db=db)

answer_prompt = PromptTemplate.from_template(
    """Given the following user question,
corresponding SQL query,
and SQL result,
answer the user question in Korean.

Question:
{question}

SQL Query:
{query}

SQL Result:
{result}

Answer:
"""
)

answer = answer_prompt | llm | StrOutputParser()

chain = (
    RunnablePassthrough.assign(query = write_query).assign(
        result=itemgetter("query") | execute_query
    )
    | answer
)

chain.invoke({"question": "주간조와 야간조의 평균 불량률을 비교해줘."})

'주간조의 평균 불량률은 1.94%이고, 야간조의 평균 불량률은 1.92%입니다. 두 조의 불량률은 거의 비슷하며, 야간조가 약간 더 낮은 불량률을 보이고 있습니다.'

In [19]:
generated_sql = write_query.invoke(
    {"question": "주간조와 야간조의 평균 불량률을 비교해줘."}
)

print(generated_sql)

SELECT
    pr.shift,
    ROUND(
        100.0 * SUM(pr.defect_qty)
        / NULLIF(SUM(pr.good_qty + pr.defect_qty), 0),
        2
    ) AS defect_rate_pct
FROM production_results AS pr
GROUP BY pr.shift;


In [24]:
from langchain_community.agent_toolkits import create_sql_agent

sql_agent_prefix = """
You are an expert SQL agent working with a {dialect} database.

Your job is to inspect the database schema, generate correct SQL,
execute the query, inspect the result, and answer the user's question.

IMPORTANT RULES:

1. Never invent tables or columns.
   Always inspect the schema before using unfamiliar tables.

2. Manufacturing defect rate is defined as:

   defect_qty / (good_qty + defect_qty)

3. For grouped defect rates such as shift, line, product, or date:

   100.0 * SUM(defect_qty)
   / NULLIF(SUM(good_qty + defect_qty), 0)

4. NEVER calculate defect rate as:

   defect_qty / good_qty

5. production_results does NOT contain line_id.

   To obtain line_id:

   production_results.order_id
   -> production_orders.order_id
   -> production_orders.line_id

6. IMPORTANT AGGREGATION RULE:

   When two or more tables each contain multiple rows
   for the same grouping key, do NOT join the raw tables
   directly before applying SUM or AVG.

   First aggregate each fact table separately to the
   requested grouping level using a CTE or subquery,
   then join the aggregated results.

   Example:
   energy_usage contains multiple rows per line,
   and production_results also contains multiple rows per line.

   WRONG:
   join energy_usage and production_results directly by line_id
   and then SUM both columns.

   CORRECT:
   aggregate energy_usage by line_id first,
   aggregate production_results by line_id first,
   then join the two aggregated results.

7. Before executing an aggregate query, check whether the JOIN
   can multiply rows and cause double counting.

8. Use explicit table aliases and qualify ambiguous columns.

9. Use NULLIF when dividing to avoid division-by-zero errors.

10. When the user asks for percentages, return percentages.

11. Unless the user specifies otherwise, limit results to at most {top_k} rows.

12. After executing the SQL, inspect whether the result magnitude
    appears reasonable. If totals appear duplicated or inflated,
    reconsider the JOIN and aggregation strategy before answering.

Always provide the final answer in Korean.
When presenting tabular results, use a Markdown table.
Use commas for large numbers and include units when appropriate.
"""

agent_executor = create_sql_agent(llm=llm, db=db, agent_type="tool-calling", prefix=sql_agent_prefix, verbose=True, top_k=10)
result = agent_executor.invoke(
    {"input": "라인별 총 전력 사용량과 총 양품 수량을 비교해줘."}
)
print(result["output"])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


alarms, employees, energy_usage, equipment, factories, maintenance_records, production_lines, production_orders, production_results, products, quality_inspections
Invoking: `sql_db_schema` with `{'table_names': 'energy_usage'}`



CREATE TABLE energy_usage (
	usage_id INTEGER, 
	line_id INTEGER NOT NULL, 
	usage_date TEXT NOT NULL, 
	electricity_kwh REAL NOT NULL, 
	compressed_air_m3 REAL NOT NULL, 
	water_m3 REAL NOT NULL, 
	PRIMARY KEY (usage_id), 
	FOREIGN KEY(line_id) REFERENCES production_lines (line_id)
)

/*
3 rows from energy_usage table:
usage_id	line_id	usage_date	electricity_kwh	compressed_air_m3	water_m3
1	1	2026-08-01	1031.65	372.28	60.91
2	2	2026-08-01	1000.41	404.6	64.23
3	3	2026-08-01	1257.84	415.17	73.06
*/
Invoking: `sql_db_schema` with `{'table_names': 'production_results'}`



CREATE TABLE production_results (
	result_id INTEGER, 
	order_id INTEGER NOT NULL, 
	work_date TEXT NOT